In [1]:
# ============================================================
# LOAN APPROVAL MODEL COMPARISON USING MLFLOW
# ============================================================

# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("sqlite:///mlflow.db")

# ============================================================
# 2. LOAD DATASET
# ============================================================

print("=" * 60)
print("STEP 1 - LOADING DATASET")
print("=" * 60)

df = pd.read_csv(r"D:\ML OPS\loan_approval_dataset.csv")

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)


# ============================================================
# 3. CLEAN COLUMN NAMES
# ============================================================

print("\n" + "=" * 60)
print("STEP 2 - CLEANING COLUMN NAMES")
print("=" * 60)

df.columns = df.columns.str.strip()
df.columns = df.columns.str.replace(" ", "_")

print("Columns:")
print(df.columns.tolist())


# ============================================================
# 4. REMOVE LOAN ID
# ============================================================

if "loan_id" in df.columns:

    df = df.drop(
        "loan_id",
        axis=1
    )

    print("\nRemoved: loan_id")


# ============================================================
# 5. DEFINE TARGET
# ============================================================

target_column = "loan_status"

if target_column not in df.columns:

    raise ValueError(
        f"Target column '{target_column}' not found."
    )

print("Target column:", target_column)


# ============================================================
# 6. CHECK MISSING VALUES
# ============================================================

print("\n" + "=" * 60)
print("STEP 3 - CHECKING MISSING VALUES")
print("=" * 60)

print(df.isnull().sum())


# ============================================================
# 7. HANDLE MISSING VALUES
# ============================================================

# Numerical columns
numeric_columns = df.select_dtypes(
    include=["int64", "float64"]
).columns

for column in numeric_columns:

    if df[column].isnull().any():

        df[column] = df[column].fillna(
            df[column].median()
        )


# Categorical columns
categorical_columns = df.select_dtypes(
    include=["object"]
).columns

for column in categorical_columns:

    if df[column].isnull().any():

        df[column] = df[column].fillna(
            df[column].mode()[0]
        )


print("\nMissing values after cleaning:")
print(df.isnull().sum())


# ============================================================
# 8. ENCODE CATEGORICAL COLUMNS
# ============================================================

print("\n" + "=" * 60)
print("STEP 4 - ENCODING CATEGORICAL VARIABLES")
print("=" * 60)

label_encoders = {}

for column in df.columns:

    if df[column].dtype == "object":

        encoder = LabelEncoder()

        df[column] = encoder.fit_transform(
            df[column].astype(str)
        )

        label_encoders[column] = encoder

        print("Encoded:", column)


print("\nCategorical variables encoded successfully.")


# ============================================================
# 9. FEATURES AND TARGET
# ============================================================

print("\n" + "=" * 60)
print("STEP 5 - PREPARING FEATURES AND TARGET")
print("=" * 60)

X = df.drop(
    target_column,
    axis=1
)

y = df[target_column]

print("Feature shape:", X.shape)
print("Target shape :", y.shape)

print("\nFeatures:")
for column in X.columns:
    print("-", column)


# ============================================================
# 10. TRAIN TEST SPLIT
# ============================================================

print("\n" + "=" * 60)
print("STEP 6 - TRAIN TEST SPLIT")
print("=" * 60)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape :", X_test.shape)


# ============================================================
# 11. CREATE MLFLOW EXPERIMENT
# ============================================================

print("\n" + "=" * 60)
print("STEP 7 - SETTING UP MLFLOW")
print("=" * 60)

experiment_name = "Loan_Approval_Model_Comparison"

mlflow.set_experiment(
    experiment_name
)

print("MLflow experiment:")
print(experiment_name)


# ============================================================
# 12. RESULTS LIST
# ============================================================

results = []


# ============================================================
# 13. MODEL TRAINING FUNCTION
# ============================================================

def train_and_evaluate(model, model_name):

    print("\n")
    print("=" * 60)
    print(f"TRAINING: {model_name}")
    print("=" * 60)

    # --------------------------------------------------------
    # Start MLflow run
    # --------------------------------------------------------

    with mlflow.start_run(
        run_name=model_name
    ):

        print("Training model...")

        # ----------------------------------------------------
        # Train
        # ----------------------------------------------------

        model.fit(
            X_train,
            y_train
        )

        print("Training completed!")

        # ----------------------------------------------------
        # Prediction
        # ----------------------------------------------------

        print("Making predictions...")

        y_pred = model.predict(
            X_test
        )

        print("Prediction completed!")

        # ----------------------------------------------------
        # Calculate metrics
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        precision = precision_score(
            y_test,
            y_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            y_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            y_pred,
            zero_division=0
        )

        # ----------------------------------------------------
        # Log model parameter
        # ----------------------------------------------------

        mlflow.log_param(
            "model",
            model_name
        )

        # ----------------------------------------------------
        # Model-specific parameters
        # ----------------------------------------------------

        if model_name == "Logistic Regression":

            mlflow.log_param(
                "max_iter",
                model.max_iter
            )

            mlflow.log_param(
                "random_state",
                42
            )

        elif model_name == "Decision Tree":

            mlflow.log_param(
                "max_depth",
                model.max_depth
            )

            mlflow.log_param(
                "random_state",
                model.random_state
            )

        elif model_name == "Random Forest":

            mlflow.log_param(
                "n_estimators",
                model.n_estimators
            )

            mlflow.log_param(
                "random_state",
                model.random_state
            )

        # ----------------------------------------------------
        # Log metrics
        # ----------------------------------------------------

        mlflow.log_metric(
            "accuracy",
            accuracy
        )

        mlflow.log_metric(
            "precision",
            precision
        )

        mlflow.log_metric(
            "recall",
            recall
        )

        mlflow.log_metric(
            "f1_score",
            f1
        )

        # ----------------------------------------------------
        # Log model
        # ----------------------------------------------------

        print("Logging model to MLflow...")

        mlflow.sklearn.log_model(
            sk_model=model,
            name="model"
        )

        print("Model logged successfully!")

        # ----------------------------------------------------
        # Store results
        # ----------------------------------------------------

        results.append({
            "Model": model_name,
            "Accuracy": accuracy,
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1
        })

        # ----------------------------------------------------
        # Display metrics
        # ----------------------------------------------------

        print("\nModel Results")
        print("-" * 40)

        print(
            f"Accuracy : {accuracy:.4f}"
        )

        print(
            f"Precision: {precision:.4f}"
        )

        print(
            f"Recall   : {recall:.4f}"
        )

        print(
            f"F1 Score : {f1:.4f}"
        )


# ============================================================
# 14. LOGISTIC REGRESSION
# ============================================================

logistic_model = LogisticRegression(
    max_iter=1000
)

train_and_evaluate(
    logistic_model,
    "Logistic Regression"
)


# ============================================================
# 15. DECISION TREE
# ============================================================

decision_tree_model = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

train_and_evaluate(
    decision_tree_model,
    "Decision Tree"
)


# ============================================================
# 16. RANDOM FOREST
# ============================================================

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

train_and_evaluate(
    random_forest_model,
    "Random Forest"
)


# ============================================================
# 17. CREATE COMPARISON DATAFRAME
# ============================================================

print("\n")
print("=" * 75)
print("FINAL MODEL COMPARISON")
print("=" * 75)

results_df = pd.DataFrame(
    results
)


# ============================================================
# 18. DISPLAY RESULTS
# ============================================================

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 19. SAVE RESULTS
# ============================================================

results_df.to_csv(
    "loan_model_comparison_results.csv",
    index=False
)

print("\nResults saved to:")
print("loan_model_comparison_results.csv")


# ============================================================
# 20. FINAL MESSAGE
# ============================================================

print("\n" + "=" * 75)
print("ALL MODELS TRAINED SUCCESSFULLY!")
print("=" * 75)

print("\nModels:")
print("1. Logistic Regression")
print("2. Decision Tree")
print("3. Random Forest")

print("\nMLflow Experiment:")
print(experiment_name)

print("\nExperiment tracking completed.")

STEP 1 - LOADING DATASET
Dataset loaded successfully!
Dataset shape: (4269, 13)

STEP 2 - CLEANING COLUMN NAMES
Columns:
['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']

Removed: loan_id
Target column: loan_status

STEP 3 - CHECKING MISSING VALUES
no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term                   0
cibil_score                 0
residential_assets_value    0
commercial_assets_value     0
luxury_assets_value         0
bank_asset_value            0
loan_status                 0
dtype: int64

Missing values after cleaning:
no_of_dependents            0
education                   0
self_employed               0
income_annum                0
loan_amount                 0
loan_term  